# H3: Joint Fitness Model Comparison

M4 (joint W with ω + κ) vs M1 (effort-only), M2 (threat-only), M3 (single-parameter).

All models compared on joint (choice + vigor) WAIC and PSIS-LOO.
Results read from `results/stats/joint_optimal/mcmc_model_comparison.csv`.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
from config import *

# Load MCMC results for both samples
results = {}
for name, s in SAMPLES.items():
    mc_path = s.model_dir / 'mcmc_model_comparison.csv'
    if mc_path.exists():
        mc = pd.read_csv(mc_path).set_index('Model')
        results[name] = mc
        print(f'{s.label}:')
        print(mc[['WAIC','LOO','choice_acc','vigor_r2','converged']].to_string())
        print()


## H3a-c: Model Comparisons (Both Samples)

In [ ]:
# H3a-c: Run all comparisons for both samples
h3_results = []

for sample_name, mc in results.items():
    label = SAMPLES[sample_name].label
    if 'M4' not in mc.index:
        print(f'{label}: M4 not found — skipping')
        continue
    
    m4_waic = mc.loc['M4', 'WAIC']
    m4_loo = mc.loc['M4', 'LOO']
    
    print(f'\n{label}:')
    print(f'  M4: WAIC={m4_waic:.0f}, LOO={m4_loo:.0f}, acc={mc.loc["M4","choice_acc"]:.3f}, vig r²={mc.loc["M4","vigor_r2"]:.3f}')
    print()
    
    for alt, h_name in [('M1', 'H3a'), ('M2', 'H3b'), ('M3', 'H3c')]:
        if alt not in mc.index: continue
        dw = mc.loc[alt, 'WAIC'] - m4_waic
        dl = mc.loc[alt, 'LOO'] - m4_loo if not np.isnan(mc.loc[alt, 'LOO']) else np.nan
        conv = mc.loc[alt, 'converged']
        agree = dw > 0 and (np.isnan(dl) or dl > 0)
        
        dl_str = f'ΔLOO={dl:+.0f}' if not np.isnan(dl) else 'LOO=N/A'
        conv_str = '' if conv else ' [non-convergent]'
        verdict = 'CONFIRMED' if agree else 'EQUIVOCAL' if dw > 0 else 'FAILED'
        
        print(f'  {h_name}: M4 vs {alt} — ΔWAIC={dw:+.0f}, {dl_str} → {verdict}{conv_str}')
        h3_results.append({'sample': sample_name, 'test': h_name, 'alt': alt,
                           'dWAIC': dw, 'dLOO': dl, 'pass': agree, 'converged': conv})

In [ ]:
# H3 Summary — both samples
print('H3 SUMMARY')
print('=' * 70)
print(f'{"Test":<8} {"Alt":<6}', end='')
for name in results: print(f'  {SAMPLES[name].label[:15]:>18}', end='')
print()
print('-' * 70)

for test, alt in [('H3a','M1'), ('H3b','M2'), ('H3c','M3')]:
    print(f'{test:<8} {alt:<6}', end='')
    for name, mc in results.items():
        if 'M4' in mc.index and alt in mc.index:
            dw = mc.loc[alt, 'WAIC'] - mc.loc['M4', 'WAIC']
            dl = mc.loc[alt, 'LOO'] - mc.loc['M4', 'LOO'] if not np.isnan(mc.loc[alt, 'LOO']) else np.nan
            agree = dw > 0 and (np.isnan(dl) or dl > 0)
            conv = '*' if not mc.loc[alt, 'converged'] else ''
            print(f'  Δ={dw:>+5.0f}{conv} {"✓" if agree else "✗"}', end='  ')
        else:
            print(f'  {"N/A":>12}', end='  ')
    print()

print('\n* = non-convergent (result still decisive)')